# Logit Lens: Tracking Answer Emergence

The logit lens applies the model's final unembedding layer to intermediate hidden states, revealing what the model "thinks" at each layer before producing the final output.

## What You'll Learn

1. Applying the logit lens to hidden states
2. Tracking when correct answers emerge across layers
3. Analyzing prediction confidence evolution
4. Measuring prediction stability and convergence
5. Identifying critical layers for reasoning

**Estimated time**: 15 minutes  
**Prerequisites**: Complete notebooks 01 and 02

## Setup

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import torch
import plotly.graph_objects as go
import numpy as np
from interpretability import load_model
from interpretability.extraction import (
    extract_hidden_states,
    apply_logit_lens,
    find_answer_emergence_layer,
    track_token_rank,
    get_prediction_entropy,
    get_prediction_stability,
    find_convergence_layer
)

print("✓ Imports successful")

/Users/wkang/code1/reasoning_explore/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-19 16:54:08.522 | DEBUG    | interpretability.models.registry:register:54 - Registered loader 'deepseek-r1-distill': DeepSeekLoader
2026-04-19 16:54:08.522 | DEBUG    | interpretability.models.registry:register:54 - Registered loader 'deepseek': DeepSeekLoader
2026-04-19 16:54:08.522 | DEBUG    | interpretability.models.registry:add_alias:65 - Added alias 'deepseek-1.5b' -> 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'
2026-04-19 16:54:08.522 | DEBUG    | interpretability.models.registry:add_alias:65 - Added alias 'deepseek-7b' -> 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B'
2026-04-19 16:54:08.523 | DEBUG    | interpretability.models.registry:add_alias:65 - Added alias 'deepseek-8b' -> 'deepseek-ai/DeepSeek-R1

✓ Imports successful


## 1. Load Model and Prepare Input

In [2]:
# Load model with platform-appropriate settings
from interpretability import get_recommended_device

device = get_recommended_device()
print(f"Using device: {device}")

model = load_model("deepseek-1.5b", device=device, quantization=None)
print(model)

2026-04-19 16:54:08.538 | DEBUG    | interpretability.models.registry:get_loader:93 - Matched 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B' to loader 'deepseek-r1-distill'
2026-04-19 16:54:08.539 | INFO     | interpretability.models.deepseek:load:61 - Loading DeepSeek model: deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
2026-04-19 16:54:08.539 | DEBUG    | interpretability.models.deepseek:get_recommended_quantization:216 - macOS detected - recommending no quantization
2026-04-19 16:54:08.539 | INFO     | interpretability.core.model_wrapper:_load_model:142 - Loading model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B...
2026-04-19 16:54:08.539 | DEBUG    | interpretability.core.model_wrapper:_load_model:156 - Using 'eager' attention implementation for interpretability


Using device: mps


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 339/339 [00:01<00:00, 227.75it/s]
2026-04-19 16:54:10.612 | DEBUG    | interpretability.core.model_wrapper:_load_model:176 - Enabled output_attentions in model config
2026-04-19 16:54:10.613 | DEBUG    | interpretability.core.model_wrapper:_load_model:181 - Enabled output_hidden_states in model config
2026-04-19 16:54:10.613 | INFO     | interpretability.core.model_wrapper:_load_model:183 - Loaded model on mps with no quantization
2026-04-19 16:54:11.687 | DEBUG    | interpretability.core.model_wrapper:_load_tokenizer:205 - Loaded tokenizer for deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
2026-04-19 16:54:11.688 | DEBUG    | interpretability.core.model_wrapper:_initialize_metadata:233 - Model metadata: 28 layers, 12 heads, 1536 hidden size
2026-04-19 16:54:11.688 | INFO     | interpretability.core.activation_cache:__init__:55 - Initialized ActivationCache with max size 1000 MB
2026-04-19 16:54:11.689 | INF

ModelWrapper(
  model=deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B,
  layers=28,
  params=1777.1M,
  size=6779.1MB,
  device=mps:0,
  quantization=none
)


In [3]:
# Math reasoning prompt
prompt = "What is 15 * 8? Let me calculate: 15 * 8 = 120"

inputs = model.tokenize(prompt, move_to_device=True)
tokens = model.tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

print(f"Tokens ({len(tokens)}):")
for i, token in enumerate(tokens):
    print(f"  {i:2d}: {token}")

Tokens (24):
   0: What
   1: Ġis
   2: Ġ
   3: 1
   4: 5
   5: Ġ*
   6: Ġ
   7: 8
   8: ?
   9: ĠLet
  10: Ġme
  11: Ġcalculate
  12: :
  13: Ġ
  14: 1
  15: 5
  16: Ġ*
  17: Ġ
  18: 8
  19: Ġ=
  20: Ġ
  21: 1
  22: 2
  23: 0


## 2. Extract Hidden States

In [4]:
# Extract hidden states from all layers
hidden_states = extract_hidden_states(
    model,
    inputs['input_ids'],
    attention_mask=inputs['attention_mask'],
    include_tokens=True
)

print(f"Hidden states shape: {hidden_states.states.shape}")
print(f"  Layers: {hidden_states.num_layers}")
print(f"  Sequence length: {hidden_states.seq_len}")
print(f"  Hidden dimension: {hidden_states.hidden_dim}")

2026-04-19 16:54:11.710 | DEBUG    | interpretability.extraction.hidden_states:extract_hidden_states:37 - Extracting hidden states from 1 sequences
2026-04-19 16:54:12.079 | INFO     | interpretability.extraction.hidden_states:extract_hidden_states:72 - Extracted hidden states: 29 layers, 24 tokens, 1536 dims


Hidden states shape: torch.Size([29, 1, 24, 1536])
  Layers: 29
  Sequence length: 24
  Hidden dimension: 1536


## 3. Apply Logit Lens

The logit lens reveals what tokens the model predicts at each layer.

In [5]:
# Apply logit lens
logit_lens_results = apply_logit_lens(
    model,
    hidden_states,
    top_k=10,
    normalize=True
)

print(f"Logit lens applied to {logit_lens_results.num_layers} layers")
print(f"Top-k predictions per position: 10")

2026-04-19 16:54:12.083 | DEBUG    | interpretability.extraction.logit_lens:apply_logit_lens:34 - Applying logit lens with top_k=10
2026-04-19 16:54:14.166 | INFO     | interpretability.extraction.logit_lens:apply_logit_lens:100 - Applied logit lens across 29 layers


Logit lens applied to 29 layers
Top-k predictions per position: 10


## 4. Examine Predictions at Different Layers

Let's look at what the model predicts at the last token position across layers.

In [6]:
# Look at last token predictions across layers
position_idx = -1  # Last position

print("Top predictions at last position across selected layers:\n")
for layer_idx in [0, 5, 10, 15, 20, 25, hidden_states.num_layers-1]:
    if layer_idx >= hidden_states.num_layers:
        continue
    
    preds = logit_lens_results.get_layer_predictions(layer_idx, batch_idx=0)
    top_preds = preds[position_idx][:5]  # Top 5
    
    print(f"Layer {layer_idx:2d}:")
    for rank, (token, prob, token_id) in enumerate(top_preds):
        print(f"  {rank+1}. '{token}' ({prob:.3f})")
    print()

Top predictions at last position across selected layers:

Layer  0:
  1. '0' (1.000)
  2. '1' (0.000)
  3. '2' (0.000)
  4. '3' (0.000)
  5. '4' (0.000)

Layer  5:
  1. 'th' (0.095)
  2. 'Ġpet' (0.028)
  3. 'Ġdifferent' (0.014)
  4. 'âĢ³' (0.011)
  5. 'Ġtimes' (0.009)

Layer 10:
  1. 'Ġdifferent' (0.207)
  2. 'Ġand' (0.075)
  3. 'Ġbasic' (0.029)
  4. 'Ġseries' (0.016)
  5. 'th' (0.014)

Layer 15:
  1. '.' (0.025)
  2. 'th' (0.013)
  3. 'Ġout' (0.011)
  4. 'Ġextra' (0.009)
  5. '-faced' (0.006)

Layer 20:
  1. '.' (0.196)
  2. 'Ġbecause' (0.054)
  3. '.Ċ' (0.051)
  4. '.ĊĊ' (0.024)
  5. '$.' (0.008)

Layer 25:
  1. '.' (0.560)
  2. '.Ċ' (0.390)
  3. '.ĊĊ' (0.029)
  4. '?' (0.005)
  5. 'Ġbecause' (0.005)

Layer 28:
  1. '.' (0.631)
  2. '.ĊĊ' (0.088)
  3. 'Ġand' (0.050)
  4. 'Ġor' (0.043)
  5. 'Ġis' (0.037)



## 5. Find When Answer Emerges

At which layer does the correct answer first appear in top predictions?

In [7]:
# Find token ID for "120"
answer_str = "120"
answer_tokens = model.tokenizer.encode(answer_str, add_special_tokens=False)
print(f"Answer '{answer_str}' token IDs: {answer_tokens}")

if answer_tokens:
    target_token_id = answer_tokens[0]
    
    # Find emergence layer
    emergence_layer = find_answer_emergence_layer(
        logit_lens_results,
        target_token=target_token_id,
        threshold_rank=5,
        position_idx=-1
    )
    
    if emergence_layer >= 0:
        print(f"✨ Answer token '{answer_str}' first appeared in top-5 at layer {emergence_layer}")
    else:
        print(f"⚠️  Answer token '{answer_str}' never appeared in top-5")
else:
    print("Could not tokenize answer")

2026-04-19 16:54:14.174 | DEBUG    | interpretability.extraction.logit_lens:find_answer_emergence_layer:132 - Target token 16 emerged at layer 0 (rank 1, prob 0.0000)


Answer '120' token IDs: [16, 17, 15]
✨ Answer token '120' first appeared in top-5 at layer 0


## 6. Track Token Rank Evolution

Let's track how the rank and probability of the answer token evolve.

In [8]:
if answer_tokens:
    # Track rank across layers
    rank_evolution = track_token_rank(
        logit_lens_results,
        target_token=target_token_id,
        position_idx=-1
    )
    
    # Extract ranks and probabilities
    ranks = []
    probs = []
    layers = []
    
    for layer_idx, result in enumerate(rank_evolution):
        if result is not None:
            rank, prob = result
            ranks.append(rank)
            probs.append(prob)
            layers.append(layer_idx)
    
    # Plot evolution
    fig = go.Figure()
    
    # Rank evolution (inverted y-axis so rank 1 is at top)
    fig.add_trace(go.Scatter(
        x=layers,
        y=ranks,
        mode='lines+markers',
        name='Rank',
        yaxis='y',
        line=dict(color='blue'),
        hovertemplate='Layer: %{x}<br>Rank: %{y}<extra></extra>'
    ))
    
    # Probability evolution
    fig.add_trace(go.Scatter(
        x=layers,
        y=probs,
        mode='lines+markers',
        name='Probability',
        yaxis='y2',
        line=dict(color='red'),
        hovertemplate='Layer: %{x}<br>Prob: %{y:.3f}<extra></extra>'
    ))
    
    fig.update_layout(
        title=f"Answer Token '{answer_str}' Rank and Probability Evolution",
        xaxis_title="Layer",
        yaxis=dict(title="Rank (lower is better)", autorange='reversed'),
        yaxis2=dict(title="Probability", overlaying='y', side='right'),
        width=900,
        height=500
    )
    
    fig.show()
    
    print(f"\nAnswer token appears in top-10 predictions at {len(layers)} layers")

2026-04-19 16:54:14.178 | DEBUG    | interpretability.extraction.logit_lens:track_token_rank:176 - Tracked token 16 across 29 layers



Answer token appears in top-10 predictions at 1 layers


## 7. Prediction Entropy Analysis

Entropy measures prediction uncertainty. High entropy = model is uncertain.

In [9]:
# Compute entropy at last position
entropy_evolution = get_prediction_entropy(
    logit_lens_results,
    position_idx=-1
)

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(len(entropy_evolution))),
    y=entropy_evolution,
    mode='lines+markers',
    line=dict(color='purple', width=2),
    marker=dict(size=6),
    hovertemplate='Layer: %{x}<br>Entropy: %{y:.3f} bits<extra></extra>'
))

fig.update_layout(
    title="Prediction Entropy Across Layers",
    xaxis_title="Layer",
    yaxis_title="Entropy (bits)",
    width=900,
    height=500
)

fig.show()

print("\nInterpretation:")
print("  📈 High entropy = Model is uncertain, predictions are diffuse")
print("  📉 Low entropy = Model is confident, predictions are focused")
print(f"\nEntropy at layer 0: {entropy_evolution[0]:.3f} bits")
print(f"Entropy at final layer: {entropy_evolution[-1]:.3f} bits")

2026-04-19 16:54:14.276 | DEBUG    | interpretability.extraction.logit_lens:get_prediction_entropy:210 - Computed prediction entropy across 29 layers



Interpretation:
  📈 High entropy = Model is uncertain, predictions are diffuse
  📉 Low entropy = Model is confident, predictions are focused

Entropy at layer 0: 0.000 bits
Entropy at final layer: 1.780 bits


## 8. Prediction Stability

How stable are predictions across consecutive layers?

In [10]:
# Compute stability
stability = get_prediction_stability(
    logit_lens_results,
    position_idx=-1,
    window_size=3
)

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(len(stability))),
    y=stability,
    mode='lines+markers',
    line=dict(color='green', width=2),
    fill='tozeroy',
    fillcolor='rgba(0,255,0,0.1)',
    hovertemplate='Layer: %{x}<br>Stability: %{y:.3f}<extra></extra>'
))

# Add threshold line
fig.add_hline(
    y=0.8,
    line_dash="dash",
    line_color="red",
    annotation_text="Convergence threshold"
)

fig.update_layout(
    title="Prediction Stability Across Layers",
    xaxis_title="Layer",
    yaxis_title="Stability Score",
    width=900,
    height=500
)

fig.show()

print("\nInterpretation:")
print("  High stability = Predictions are consistent across layers")
print("  Low stability = Predictions are still changing")

2026-04-19 16:54:14.283 | DEBUG    | interpretability.extraction.logit_lens:compare_predictions_across_layers:249 - Prediction overlap between layers 0 and 3: 0.00%
2026-04-19 16:54:14.284 | DEBUG    | interpretability.extraction.logit_lens:compare_predictions_across_layers:249 - Prediction overlap between layers 1 and 3: 25.00%
2026-04-19 16:54:14.284 | DEBUG    | interpretability.extraction.logit_lens:compare_predictions_across_layers:249 - Prediction overlap between layers 2 and 3: 42.86%
2026-04-19 16:54:14.285 | DEBUG    | interpretability.extraction.logit_lens:compare_predictions_across_layers:249 - Prediction overlap between layers 1 and 4: 42.86%
2026-04-19 16:54:14.285 | DEBUG    | interpretability.extraction.logit_lens:compare_predictions_across_layers:249 - Prediction overlap between layers 2 and 4: 42.86%
2026-04-19 16:54:14.285 | DEBUG    | interpretability.extraction.logit_lens:compare_predictions_across_layers:249 - Prediction overlap between layers 3 and 4: 42.86%
2026-


Interpretation:
  High stability = Predictions are consistent across layers
  Low stability = Predictions are still changing


In [11]:
# Find convergence layer
convergence_layer = find_convergence_layer(
    logit_lens_results,
    position_idx=-1,
    stability_threshold=0.8,
    window_size=5
)

if convergence_layer >= 0:
    print(f"✨ Predictions converged at layer {convergence_layer}")
    print(f"   This means the model's answer is stable from layer {convergence_layer} onwards.")
else:
    print("⚠️  Predictions never fully converged")

2026-04-19 16:54:14.306 | DEBUG    | interpretability.extraction.logit_lens:compare_predictions_across_layers:249 - Prediction overlap between layers 0 and 5: 0.00%
2026-04-19 16:54:14.307 | DEBUG    | interpretability.extraction.logit_lens:compare_predictions_across_layers:249 - Prediction overlap between layers 1 and 5: 42.86%
2026-04-19 16:54:14.307 | DEBUG    | interpretability.extraction.logit_lens:compare_predictions_across_layers:249 - Prediction overlap between layers 2 and 5: 42.86%
2026-04-19 16:54:14.307 | DEBUG    | interpretability.extraction.logit_lens:compare_predictions_across_layers:249 - Prediction overlap between layers 3 and 5: 42.86%
2026-04-19 16:54:14.307 | DEBUG    | interpretability.extraction.logit_lens:compare_predictions_across_layers:249 - Prediction overlap between layers 4 and 5: 100.00%
2026-04-19 16:54:14.308 | DEBUG    | interpretability.extraction.logit_lens:compare_predictions_across_layers:249 - Prediction overlap between layers 1 and 6: 42.86%
2026

⚠️  Predictions never fully converged


## 9. Compare Multiple Positions

Let's compare logit lens results at different token positions.

In [12]:
# Pick interesting positions
positions = {
    "First token": 0,
    "Middle token": len(tokens) // 2,
    "Last token": -1
}

fig = go.Figure()

for label, pos in positions.items():
    entropy = get_prediction_entropy(logit_lens_results, position_idx=pos)
    
    fig.add_trace(go.Scatter(
        x=list(range(len(entropy))),
        y=entropy,
        mode='lines',
        name=label,
        hovertemplate=f'{label}<br>Layer: %{{x}}<br>Entropy: %{{y:.3f}}<extra></extra>'
    ))

fig.update_layout(
    title="Prediction Entropy at Different Token Positions",
    xaxis_title="Layer",
    yaxis_title="Entropy (bits)",
    width=900,
    height=500,
    showlegend=True
)

fig.show()

2026-04-19 16:54:14.328 | DEBUG    | interpretability.extraction.logit_lens:get_prediction_entropy:210 - Computed prediction entropy across 29 layers
2026-04-19 16:54:14.329 | DEBUG    | interpretability.extraction.logit_lens:get_prediction_entropy:210 - Computed prediction entropy across 29 layers
2026-04-19 16:54:14.330 | DEBUG    | interpretability.extraction.logit_lens:get_prediction_entropy:210 - Computed prediction entropy across 29 layers


## 10. Key Insights Summary

Let's summarize the key findings from our logit lens analysis.

In [13]:
print("=" * 60)
print("LOGIT LENS ANALYSIS SUMMARY")
print("=" * 60)

print(f"\nPrompt: \"{prompt}\"")
print(f"Target answer: '{answer_str}'")

if answer_tokens and emergence_layer >= 0:
    print(f"\n📊 Key Findings:")
    print(f"  • Answer emerges at layer: {emergence_layer} / {hidden_states.num_layers}")
    print(f"  • Answer emergence: {emergence_layer / hidden_states.num_layers:.1%} through the model")

if convergence_layer >= 0:
    print(f"  • Predictions converge at layer: {convergence_layer}")
    print(f"  • Convergence point: {convergence_layer / hidden_states.num_layers:.1%} through the model")

print(f"\n📈 Entropy:")
print(f"  • Initial entropy: {entropy_evolution[0]:.3f} bits")
print(f"  • Final entropy: {entropy_evolution[-1]:.3f} bits")
entropy_change = entropy_evolution[-1] - entropy_evolution[0]
change_label = "increase" if entropy_change > 0 else "decrease"
print(f"  • Entropy {change_label}: {abs(entropy_change):.3f} bits")

print(f"\n🎯 Interpretation:")
if emergence_layer >= 0 and emergence_layer < hidden_states.num_layers // 2:
    print("  • Early emergence: Model figured out the answer quickly")
elif emergence_layer >= hidden_states.num_layers // 2:
    print("  • Late emergence: Model needed many layers to compute answer")

if entropy_evolution[-1] < 2.0:
    print("  • Low final entropy: Model is confident in its answer")
else:
    print("  • High final entropy: Model is uncertain")

print("\n" + "=" * 60)

LOGIT LENS ANALYSIS SUMMARY

Prompt: "What is 15 * 8? Let me calculate: 15 * 8 = 120"
Target answer: '120'

📊 Key Findings:
  • Answer emerges at layer: 0 / 29
  • Answer emergence: 0.0% through the model

📈 Entropy:
  • Initial entropy: 0.000 bits
  • Final entropy: 1.780 bits
  • Entropy increase: 1.780 bits

🎯 Interpretation:
  • Early emergence: Model figured out the answer quickly
  • Low final entropy: Model is confident in its answer



## Summary

In this notebook, you learned how to:

1. ✅ Apply the logit lens to reveal intermediate predictions
2. ✅ Track when correct answers emerge across layers
3. ✅ Analyze prediction confidence and entropy
4. ✅ Measure prediction stability and convergence
5. ✅ Identify critical layers for reasoning tasks

## Key Insights

- **The logit lens reveals when answers form**: You can see the exact layer where the model "figures out" the answer
- **Entropy tracks confidence**: High entropy early, low entropy late = model becomes more certain
- **Stability indicates convergence**: Once predictions stabilize, later layers mainly refine
- **Different tasks have different emergence patterns**: Simple math vs complex reasoning show different layer-wise evolution

## Next Steps

- **Notebook 03**: Circuit discovery to find which components compute the answer
- **Notebook 05**: Comparative analysis across different prompts

---

**Experiment Ideas**:
- Try different reasoning tasks (logic, math, commonsense)
- Compare correct vs incorrect reasoning traces
- Find at what layer different types of knowledge emerge